In [1]:
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_nvidia_ai_endpoints import ChatNVIDIA
import os
from logger import Colors, log_error, log_header, log_info, log_success, log_warning



load_dotenv(override=True) 

True

In [2]:

from langchain_openai import ChatOpenAI

In [3]:
nvidia_api_key = os.getenv("NVIDIA_API_KEY")

if nvidia_api_key is None:
    log_warning("NVIDIA API Key not found in environment variables.")
else:
    log_success(f"NVIDIA API Key loaded successfully: {nvidia_api_key[:4]}...")


# tavily_api_key = os.getenv("TAVILY_API_KEY")

# if tavily_api_key is None:
#     log_warning("Tavily API Key not found in environment variables.")
# else:
#     log_success(f"Tavily API Key loaded successfully: {tavily_api_key[:4]}...")


✅ NVIDIA API Key loaded successfully: nvap...


In [4]:
log_error("This is an error message.")
log_warning("This is a warning message.")
log_info("This is an info message.")
log_success("This is a success message.")
log_header("This is a header message.")

❌ This is an error message.
⚠️  This is a warning message.
ℹ️  This is an info message.
✅ This is a success message.

🚀 This is a header message.



In [5]:
reflection_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a viral twiiter user who is reflecting on your recent tweets. You want to analyze which of your tweets have been performing well and which ones haven't. You will be given a tweet and you need to evaluate it and provide feedback."),
    MessagesPlaceholder(variable_name="messages"),
]) 

In [6]:
generation_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a twitter tech influencer who is creating viral tweets about AI. You want to create engaging and informative tweets that will resonate with your audience. You will be given a topic and you need to generate a tweet about it. If user provides a critique, respond with a revised version of the tweet that addresses the critique."),
    MessagesPlaceholder(variable_name="messages"),
])

In [7]:
MODEL = ChatNVIDIA(model="moonshotai/kimi-k2-instruct-0905")

generate_chain = generation_prompt | MODEL
reflect_chain = reflection_prompt | MODEL

In [8]:
class MessageGraph(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [9]:
REFLECT = "reflect"
GENERATE = "generate"


def generation_node(state: MessageGraph):
    return {"messages": [generate_chain.invoke({"messages": state["messages"]})]}


def reflection_node(state: MessageGraph):
    res = reflect_chain.invoke({"messages": state["messages"]})
    return {"messages": [HumanMessage(content=res.content)]}

In [10]:

def should_continue(state: MessageGraph):
    if len(state["messages"]) > 6:
        return END
    return REFLECT

In [11]:
builder = StateGraph(state_schema=MessageGraph)
builder.add_node(GENERATE, generation_node)
builder.add_node(REFLECT, reflection_node)
builder.set_entry_point(GENERATE)




builder.add_conditional_edges(GENERATE, should_continue, path_map={END: END, REFLECT: REFLECT})
builder.add_edge(REFLECT, GENERATE)

graph = builder.compile()
graph.get_graph().draw_mermaid_png(output_file_path="flow_reflection_agent.png")


b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x00\xe1\x00\x00\x00\xf9\x08\x02\x00\x00\x00^T\xc5\xdb\x00\x00\x10\x00IDATx\x9c\xec\x9d\x07|S\xd5\xdb\xc7\xcf\xbd\x19M\xf7\x1e\x94\xd2\x06(\x952\x0b\x94\n\xca\x1e\x82\xa0\x80\xc8\x9f\xbde\xaa(K\x94\xf5*K\x90%\x02\xb2\xf7(e)\x9b\xb2Qd\x95]\xf6*tA\x0b\xddMW\xc6\xcd\xfb$\xb7MC\x9b\x94\x14\x9a\xe4$\xf7|\xe5\x13o\xce9\xf7$\xbd\xf9\xdds\x9e\xe79\xe3\xf2\x95J%"\x100\x86\x8f\x08\x04\xbc!\x1a%\xe0\x0e\xd1(\x01w\x88F\t\xb8C4J\xc0\x1d\xa2Q\x02\xee\x10\x8d\xea\xe5\xfa\x99\xcc\x171y\xb9\x99r\xb9\x8c\x91\xe5\x17G\xe8\x94\x94\xea?\x8a\xa7T\x1d1Eoi%\x03\xafJD\xf3\x11#W\x97\xa3T\x05(\x1a\xc1\xff\x10S|\xa0\xca\xe1!\x86a\xcfBPCaI\x1eR*\x10\xd4\xa3z\xcf\xa8?\x08)i\xba\xf0\x18\xd1\xf0^U\xac\xf0KP\x88\xa2\xe0\\F\x9dQ\x08-\xa4x4%\x14\xd1\xee\xbe\xc2Z\xa1\xce\xdeU\x85\xc8*\xa0H|\xb4\x04G7%\xbfx\x9a\x9b\x9f\xab\xe0\x0bh\xf8\xbd\x05B\x1ad\'/`4\x05@7\xa00\x9e\x80b\x14J\x10\x10E\xa9\xaea\xa1\xda\xe0\x82\xf2\x91\xb2P\xa3*\x95Q*\tQ*\xfd\x82\xda E\xa1\xba\xdap\x0c\x87*\x9

In [12]:
print("Hello LangGraph")
inputs = {
        "messages": [
            HumanMessage(
                content="""Make this tweet better:"
                                    @LangChainAI
            — newly Tool Calling feature is seriously underrated.

            After a long wait, it's  here- making the implementation of agents across different models with function calling - super easy.

            Made a video covering their newest blog post

                                  """
            )
        ]
    }
response = graph.invoke(inputs)
print(response)

Hello LangGraph
{'messages': [HumanMessage(content='Make this tweet better:"\n                                    @LangChainAI\n            — newly Tool Calling feature is seriously underrated.\n\n            After a long wait, it\'s  here- making the implementation of agents across different models with function calling - super easy.\n\n            Made a video covering their newest blog post\n\n                                  ', additional_kwargs={}, response_metadata={}, id='ad44a12b-9141-490b-8736-3b39bd8c3fa2'), AIMessage(content='BREAKING: @LangChainAI just dropped Tool Calling and it’s the cheat-code every dev was waiting for.  \n1️⃣ One schema → works with GPT-4, Claude, Gemini  \n2️⃣ No more prompt-tuning hell  \n3️⃣ Ship an agent in 20 lines  \n\nI速度和 translated their new blog into a 60-sec build here 👇', additional_kwargs={}, response_metadata={'role': 'assistant', 'content': 'BREAKING: @LangChainAI just dropped Tool Calling and it’s the cheat-code every dev was waiting fo

In [15]:
print(response["messages"][0].content)

Make this tweet better:"
                                    @LangChainAI
            — newly Tool Calling feature is seriously underrated.

            After a long wait, it's  here- making the implementation of agents across different models with function calling - super easy.

            Made a video covering their newest blog post

                                  
